In [1]:
import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
!pip install dagshub mlflow optuna -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 595.2 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 2.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 33.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from kaggle_secrets import UserSecretsClient
import warnings
warnings.filterwarnings('ignore')

dagshub.init (
    repo_owner="sophyrise",
    repo_name='ieee-cis-fraud-detection',
    mlflow=True
)

mlflow.set_experiment("LogisticRegression-Training")
print("✅ MLflow tracking URI:", mlflow.get_tracking_uri())

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=bcef3b5e-3dcd-4e6a-bb72-e6538700fc33&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=71a9a99b3fdfc41648f18c028ffafc9f0789f7131f4d768a66a1d6166008919a




Accessing as sophyrise

Initialized MLflow to track repo "sophyrise/ieee-cis-fraud-detection"

Repository sophyrise/ieee-cis-fraud-detection initialized!

✅ MLflow tracking URI: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow


# **Cleaning**

In [4]:
with mlflow.start_run(run_name="LogisticRegression_Cleaning"):
    import gc
    import numpy as np
    import pandas as pd

    DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"
    TXN_MISSING_THRESHOLD = 0.80
    ID_MISSING_THRESHOLD = 0.95
    NEAR_CONSTANT_THRESHOLD = 0.99

    # load
    train_txn = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
    train_id  = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
    test_txn  = pd.read_csv(f"{DATA_DIR}/test_transaction.csv")
    test_id   = pd.read_csv(f"{DATA_DIR}/test_identity.csv")

    # fix id-01 vs id_01
    test_id.columns = test_id.columns.str.replace("-", "_", regex=False)

    # merge
    train = train_txn.merge(train_id, on="TransactionID", how="left")
    test  = test_txn.merge(test_id, on="TransactionID", how="left")

    del train_txn, train_id, test_txn, test_id
    gc.collect()

    # split target
    y_train = train["isFraud"].astype(np.int8)
    X_train = train.drop(columns=["isFraud", "TransactionID"])
    X_test  = test.drop(columns=["TransactionID"])

    del train, test
    gc.collect()

    # drop high-missing
    id_like_cols = [c for c in X_train.columns if c.startswith("id_") or c in ["DeviceType", "DeviceInfo"]]
    txn_like_cols = [c for c in X_train.columns if c not in id_like_cols]
    missing_ratio = X_train.isnull().mean()

    drop_txn = [c for c in txn_like_cols if missing_ratio[c] > TXN_MISSING_THRESHOLD]
    drop_id  = [c for c in id_like_cols if missing_ratio[c] > ID_MISSING_THRESHOLD]
    drop_missing = drop_txn + drop_id

    X_train = X_train.drop(columns=drop_missing)
    X_test  = X_test.drop(columns=[c for c in drop_missing if c in X_test.columns])

    # drop near-constant
    near_constant_cols = []
    for col in X_train.columns:
        top_freq = X_train[col].value_counts(dropna=False, normalize=True).iloc[0]
        if top_freq > NEAR_CONSTANT_THRESHOLD:
            near_constant_cols.append(col)

    X_train = X_train.drop(columns=near_constant_cols)
    X_test  = X_test.drop(columns=[c for c in near_constant_cols if c in X_test.columns])

    # align test columns
    for col in X_train.columns:
        if col not in X_test.columns:
            X_test[col] = np.nan
    X_test = X_test[X_train.columns]

    # log
    mlflow.log_param("txn_missing_threshold", TXN_MISSING_THRESHOLD)
    mlflow.log_param("id_missing_threshold", ID_MISSING_THRESHOLD)
    mlflow.log_param("near_constant_threshold", NEAR_CONSTANT_THRESHOLD)

    mlflow.log_metric("train_rows", int(X_train.shape[0]))
    mlflow.log_metric("test_rows", int(X_test.shape[0]))
    mlflow.log_metric("final_features", int(X_train.shape[1]))
    mlflow.log_metric("fraud_rate", float(y_train.mean()))
    mlflow.log_metric("dropped_missing_cols", int(len(drop_missing)))
    mlflow.log_metric("dropped_near_constant_cols", int(len(near_constant_cols)))

    print(f"X_train_clean: {X_train.shape}")
    print(f"X_test_clean:  {X_test.shape}")

    # keep for next cells
    X_train_clean = X_train
    X_test_clean = X_test
    y_train_clean = y_train

X_train_clean: (590540, 353)
X_test_clean:  (506691, 353)
🏃 View run LogisticRegression_Cleaning at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/3bcd5c2db95041c1b217e648f7ea2296
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/1


# **Feature Engineering**

In [5]:
with mlflow.start_run(run_name="LogisticRegression_FeatureEngineering"):
    from sklearn.preprocessing import LabelEncoder

    X_train_fe = X_train_clean.copy()
    X_test_fe = X_test_clean.copy()
    y_fe = y_train_clean.copy()

    # making numbers less extreme so the model can learn better
    X_train_fe["TransactionAmt_log"] = np.log1p(X_train_fe["TransactionAmt"])
    X_test_fe["TransactionAmt_log"] = np.log1p(X_test_fe["TransactionAmt"])

    # extract cents. fraud often has patterns in decimals so model can recognize it easily
    X_train_fe["TransactionAmt_cents"] = (X_train_fe["TransactionAmt"] % 1).round(3)
    X_test_fe["TransactionAmt_cents"] = (X_test_fe["TransactionAmt"] % 1).round(3)

    # humans often make transactions with round numbers, otherwise it is suspicious, so we create column with 0/1 values, where with 0 is defined as more likely fraud case  
    X_train_fe["TransactionAmt_isround"] = (X_train_fe["TransactionAmt_cents"] == 0).astype(np.int8)
    X_test_fe["TransactionAmt_isround"] = (X_test_fe["TransactionAmt_cents"] == 0).astype(np.int8)

    # takes hour
    X_train_fe["hour"] = (X_train_fe["TransactionDT"] // 3600) % 24
    X_test_fe["hour"] = (X_test_fe["TransactionDT"] // 3600) % 24

    # takes day
    X_train_fe["day"] = (X_train_fe["TransactionDT"] // (3600 * 24)) % 7
    X_test_fe["day"] = (X_test_fe["TransactionDT"] // (3600 * 24)) % 7

    # maps time to circle thorugh sin and cos
    for col, period in [("hour", 24), ("day", 7)]:
        X_train_fe[f"{col}_sin"] = np.sin(2 * np.pi * X_train_fe[col] / period)
        X_train_fe[f"{col}_cos"] = np.cos(2 * np.pi * X_train_fe[col] / period)
        X_test_fe[f"{col}_sin"] = np.sin(2 * np.pi * X_test_fe[col] / period)
        X_test_fe[f"{col}_cos"] = np.cos(2 * np.pi * X_test_fe[col] / period)

    # drop original columns cause we created new, more important ones using them
    X_train_fe = X_train_fe.drop(columns=["TransactionDT", "hour", "day"])
    X_test_fe = X_test_fe.drop(columns=["TransactionDT", "hour", "day"])

    # break down email into two parts - provider and suffix
    for col in ["P_emaildomain", "R_emaildomain"]:
        if col in X_train_fe.columns:
            for df in [X_train_fe, X_test_fe]:
                df[f"{col}_provider"] = df[col].str.split(".").str[0]
                df[f"{col}_suffix"] = df[col].str.split(".").str[-1]

    # identify unusual transactions through card1
    if "card1" in X_train_fe.columns:
        card1_stats = (
            X_train_fe.groupby("card1")["TransactionAmt"]
            .agg(card1_amt_mean="mean", card1_amt_std="std", card1_count="count")
            .reset_index()
        )
        X_train_fe = X_train_fe.merge(card1_stats, on="card1", how="left")
        X_test_fe = X_test_fe.merge(card1_stats, on="card1", how="left")

        X_train_fe["card1_amt_zscore"] = (
                (X_train_fe["TransactionAmt"] - X_train_fe["card1_amt_mean"])
                / (X_train_fe["card1_amt_std"] + 1e-5)
        )
        X_test_fe["card1_amt_zscore"] = (
                (X_test_fe["TransactionAmt"] - X_test_fe["card1_amt_mean"])
                / (X_test_fe["card1_amt_std"] + 1e-5)
        )

    # loops over columns and checks if columns contain NaN, if so, it created flags(0/1)
    # so it shows 1 when the value is missing
    cols_with_missing = [c for c in X_train_fe.columns if X_train_fe[c].isnull().any()]
    for col in cols_with_missing:
        X_train_fe[f"{col}_isnan"] = X_train_fe[col].isnull().astype(np.int8)
        X_test_fe[f"{col}_isnan"] = X_test_fe[col].isnull().astype(np.int8)

    # WOE ENCODING - defines number that is connected to target without the need for many categories
    category_cols = X_train_fe.select_dtypes(include=["object"]).columns.tolist()
    HIGH_CARD_THRESHOLD = 6
    high_cardinality_cols = [c for c in category_cols if X_train_fe[c].nunique() > HIGH_CARD_THRESHOLD]
    low_cardinality_cols = [c for c in category_cols if X_train_fe[c].nunique() <= HIGH_CARD_THRESHOLD]

    woe_mappings = {}
    iv_values = {}

    for col in high_cardinality_cols:
        tmp = pd.DataFrame({"col": X_train_fe[col].astype(str),
                            "target": y_fe.values})
        grp = tmp.groupby("col")["target"].agg(["count", "mean"])
        grp["n_pos"] = grp["mean"] * grp["count"]
        grp["n_neg"] = grp["count"] - grp["n_pos"]
        total_pos = grp["n_pos"].sum()
        total_neg = grp["n_neg"].sum()
        grp["p_pos"] = grp["n_pos"] / (total_pos + 1e-8)
        grp["p_neg"] = grp["n_neg"] / (total_neg + 1e-8)
        grp["woe"] = np.log((grp["p_pos"] + 1e-8) / (grp["p_neg"] + 1e-8))
        grp["iv"] = (grp["p_pos"] - grp["p_neg"]) * grp["woe"]

        woe_mappings[col] = grp["woe"].to_dict()
        iv_values[col] = grp["iv"].sum()

        X_train_fe[f"{col}_woe"] = X_train_fe[col].astype(str).map(woe_mappings[col]).fillna(0)
        X_test_fe[f"{col}_woe"] = X_test_fe[col].astype(str).map(woe_mappings[col]).fillna(0)

    # delete original categorical columns
    X_train_fe = X_train_fe.drop(columns=high_cardinality_cols, errors="ignore")
    X_test_fe = X_test_fe.drop(columns=high_cardinality_cols, errors="ignore")

    # one-hot encoding on low-cardinality
    X_train_fe = pd.get_dummies(X_train_fe, columns=low_cardinality_cols, drop_first=True, dummy_na=True)
    X_test_fe = pd.get_dummies(X_test_fe, columns=low_cardinality_cols, drop_first=True, dummy_na=True)

    # align train and test
    X_train_fe, X_test_fe = X_train_fe.align(X_test_fe, join="left", axis=1, fill_value=0)

    # gets median for each column and fill missed values
    num_cols = X_train_fe.select_dtypes(include=[np.number]).columns
    medians = X_train_fe[num_cols].median()
    X_train_fe[num_cols] = X_train_fe[num_cols].fillna(medians)
    X_test_fe[num_cols] = X_test_fe[num_cols].fillna(medians)

    # just log everything
    mlflow.log_param("high_card_threshold", HIGH_CARD_THRESHOLD)
    mlflow.log_param("encoding_high_card", "WOE")
    mlflow.log_param("encoding_low_card", "OneHot")
    mlflow.log_param("amt_log_transform", True)
    mlflow.log_param("cyclic_time_encoding", True)
    mlflow.log_param("missing_indicator_flags", len(cols_with_missing))
    mlflow.log_param("woe_encoded_cols", len(high_cardinality_cols))
    mlflow.log_param("onehot_encoded_cols", len(low_cardinality_cols))
    mlflow.log_metric("features_before_fe", int(X_train_clean.shape[1]))
    mlflow.log_metric("features_after_fe", int(X_train_fe.shape[1]))
    mlflow.log_metric("new_features_created", int(X_train_fe.shape[1] - X_train_clean.shape[1]))

    import matplotlib.pyplot as plt
    import seaborn as sns

    # IV table picture drawing
    iv_df = (pd.DataFrame.from_dict(iv_values, orient="index", columns=["IV"])
             .sort_values("IV", ascending=False))
    plt.figure(figsize=(10, 8))
    sns.barplot(x=iv_df["IV"][:20], y=iv_df.index[:20], palette="viridis")
    plt.title("Top 20 Features by Information Value (IV)")
    plt.xlabel("Information Value")
    plt.ylabel("Features")
    plt.tight_layout()
    plt.savefig("iv_features.png", bbox_inches='tight', dpi=300)
    mlflow.log_artifact("iv_features.png")
    plt.close()
    
    
    print(f"Features before FE : {X_train_clean.shape[1]}")
    print(f"Features after  FE : {X_train_fe.shape[1]}")
    print(f"\nTop WOE features by IV:")
    print(iv_df.head(10).to_string())

Features before FE : 353
Features after  FE : 743

Top WOE features by IV:
                              IV
DeviceInfo              0.844126
R_emaildomain           0.668873
R_emaildomain_provider  0.649126
id_31                   0.605114
R_emaildomain_suffix    0.513792
P_emaildomain           0.206683
P_emaildomain_provider  0.189358
id_33                   0.140332
id_30                   0.083661
P_emaildomain_suffix    0.037712
🏃 View run LogisticRegression_FeatureEngineering at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/93d20f74051f457b9e9a3cc883003974
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/1


# **Feature Selection**

In [6]:
with mlflow.start_run(run_name="LogisticRegression_FeatureSelection"):
    X_train_fs = X_train_fe.copy()
    X_test_fs = X_test_fe.copy()

    # replacing empty spots and infinity values with 0
    X_train_fs = X_train_fs.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test_fs = X_test_fs.replace([np.inf, -np.inf], np.nan).fillna(0)

    # drop columns where predictive power is less that 0.02
    IV_THRESHOLD = 0.02
    low_iv_cols = []
    for col, iv in iv_values.items():
        if iv < IV_THRESHOLD:
            woe_col = f"{col}_woe"
            if woe_col in X_train_fs.columns:
                low_iv_cols.append(woe_col)
            elif col in X_train_fs.columns:
                low_iv_cols.append(col)

    X_train_fs = X_train_fs.drop(columns=low_iv_cols, errors="ignore")
    X_test_fs = X_test_fs.drop(columns=low_iv_cols, errors="ignore")

    # drop columns that have only one unique value
    nunique = X_train_fs.nunique(dropna=False)
    near_zero_cols = nunique[nunique <= 1].index.tolist()
    X_train_fs = X_train_fs.drop(columns=near_zero_cols, errors="ignore")
    X_test_fs = X_test_fs.drop(columns=near_zero_cols, errors="ignore")

    # drop columns that are too similar (correlation > 0.95)
    CORR_THRESHOLD = 0.95
    num_cols = X_train_fs.select_dtypes(include=[np.number]).columns.tolist()

    # Use 120k rows to speed up the math and in order not to freeze my screeen!
    sample_n = min(120_000, len(X_train_fs))
    sample_idx = np.random.RandomState(42).choice(len(X_train_fs), size=sample_n, replace=False)
    X_sample = X_train_fs.iloc[sample_idx][num_cols]

    corr_matrix = X_sample.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop_corr = [c for c in upper.columns if (upper[c] > CORR_THRESHOLD).any()]

    X_train_fs = X_train_fs.drop(columns=to_drop_corr, errors="ignore")
    X_test_fs = X_test_fs.drop(columns=to_drop_corr, errors="ignore")

    # reindex so that test matches train
    X_test_fs = X_test_fs.reindex(columns=X_train_fs.columns, fill_value=0)

    # log
    mlflow.log_param("iv_threshold", IV_THRESHOLD)
    mlflow.log_param("corr_threshold", CORR_THRESHOLD)
    mlflow.log_metric("dropped_low_iv", len(low_iv_cols))
    mlflow.log_metric("dropped_near_zero", len(near_zero_cols))
    mlflow.log_metric("dropped_high_corr", len(to_drop_corr))
    mlflow.log_metric("final_feature_count", int(X_train_fs.shape[1]))

    print(f"Dropped low IV: {len(low_iv_cols)}")
    print(f"Dropped near-zero: {len(near_zero_cols)}")
    print(f"Dropped high correlation: {len(to_drop_corr)}")
    print(f"Final features: {X_train_fs.shape[1]}")

    X_train_final = X_train_fs
    X_test_final = X_test_fs

Dropped low IV: 0
Dropped near-zero: 1
Dropped high correlation: 402
Final features: 340
🏃 View run LogisticRegression_FeatureSelection at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/e835d2527f274427b1ac54852e905b64
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/1


In [7]:
print("iv_values entries:", len(iv_values))
print("sample keys:", list(iv_values.keys())[:5])
print("sample IV values:", sorted(iv_values.values(), reverse=True)[:10])

iv_values entries: 10
sample keys: ['P_emaildomain', 'R_emaildomain', 'id_30', 'id_31', 'id_33']
sample IV values: [np.float64(0.8441262610688159), np.float64(0.6688725389897928), np.float64(0.6491255419435229), np.float64(0.6051139813174355), np.float64(0.5137920495876457), np.float64(0.20668256884739525), np.float64(0.18935778021386446), np.float64(0.14033237629334155), np.float64(0.08366083261656843), np.float64(0.03771248214371713)]
